# PrecisionPilot — 02 · T4 accuracy table + TensorRT benchmark

**Before running:**
1. *Settings → Accelerator → GPU T4* (x1 or x2; only GPU 0 is used) and *Internet → On*.
2. *Add Input* → **Notebook Output** → pick your run of `01 · Fine-tune` (it contains
   `resnet18_cifar10.pt`). Alternatively upload the checkpoint as a Kaggle Dataset.
3. *Run All*. Expected time: roughly 1–2 hours (mostly TensorRT engine builds and the
   1,024-config accuracy table).

What it does:
1. installs PrecisionPilot + TensorRT 10 and checks versions,
2. **hardware-independent accuracy** (`run-accuracy`): FP32 / FP16 / INT8 references,
   accuracy of all 1,024 configurations on the validation subset, per-unit sensitivity,
3. **T4 latency study** (`run-gpu --backend trt`): per-unit lookup table for batch 1/8/32,
   all search methods x budgets, full-engine verification, real TensorRT test accuracy,
4. builds the figures and zips `results/` for download.

**After it finishes:** download `precisionpilot_results_t4.zip` and unzip it into the repo
root on your Mac (it fills `results/accuracy/`, `results/t4/`, `results/figures/`,
`results/tables/`). The accuracy files are needed by the Windows benchmark run.

In [ ]:
!nvidia-smi

In [ ]:
import os, subprocess, sys
os.chdir("/kaggle/working")
if not os.path.exists("PrecisionPilot"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/rajat-gondkar/PrecisionPilot.git"], check=True)
os.chdir("/kaggle/working/PrecisionPilot")
subprocess.run(["git", "pull", "--ff-only"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q "tensorrt-cu12>=10.0,<11"

In [ ]:
!python -m precisionpilot env-check

In [ ]:
import glob, os, shutil
os.makedirs("checkpoints", exist_ok=True)
found = sorted(glob.glob("/kaggle/input/**/resnet18_cifar10.pt", recursive=True))
assert found, "Add the fine-tuning notebook output (or a dataset with the checkpoint) as input"
shutil.copy(found[0], "checkpoints/resnet18_cifar10.pt")
js = glob.glob(os.path.join(os.path.dirname(found[0]), "resnet18_cifar10.json"))
if js:
    shutil.copy(js[0], "checkpoints/")
print("checkpoint:", found[0])

In [ ]:
!python -m precisionpilot make-splits

In [ ]:
!python -m precisionpilot evaluate --device cuda

### Hardware-independent accuracy (reference, 1,024-config table, sensitivity)

In [ ]:
!python -m precisionpilot run-accuracy --device cuda

### T4 latency study with TensorRT

In [ ]:
!python -m precisionpilot run-gpu --device cuda --backend trt

In [ ]:
!python -m precisionpilot report

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/precisionpilot_results_t4", "zip", ".", "results")
print("download: /kaggle/working/precisionpilot_results_t4.zip")